In [1]:
# ==============================================================================
# SCRIPT 5 (FINAL ROBUST VERSION): DEBERTA ENSEMBLE WITH K-FOLD & STACKING
#
# MAJOR CORRECTIONS:
# 1. Robust Saving Logic: The script no longer saves temporary .pt files. Instead,
#    it keeps the best model's state in memory and saves the complete model
#    (weights + config) correctly in one step at the end of each fold's training.
# 2. Pre-Download Verification: A new verification step has been added. Before
#    zipping, the script checks that all required files (config.json, pytorch_model.bin)
#    exist in each fold directory and prints a confirmation. This guarantees the
#    downloadable ZIP is complete and correct.
# ==============================================================================

import os
import shutil
import joblib
import json
import pandas as pd
import numpy as np
import torch
import torch.nn.functional as F
from google.colab import files
from sklearn.model_selection import StratifiedKFold, GridSearchCV
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.svm import SVC
import lightgbm as lgb
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, accuracy_score
from torch.utils.data import DataLoader, TensorDataset
from transformers import AutoTokenizer, AutoModelForSequenceClassification, get_linear_schedule_with_warmup
from torch.optim import AdamW
from sklearn.utils.class_weight import compute_class_weight
from sklearn.feature_extraction.text import TfidfVectorizer
from scipy.sparse import hstack

# ==============================================================================
# SECTION 1: CENTRALIZED CONFIGURATION
# ==============================================================================
class Config:
    MODEL_NAME = 'microsoft/deberta-v3-base'
    OUTPUT_DIR = './student_feedback_model_ensemble_B1_G3/'
    N_SPLITS = 5
    MAX_LENGTH = 128
    BATCH_SIZE = 16
    LEARNING_RATE = 3e-5
    NUM_EPOCHS_KFOLD = 5
    EARLY_STOPPING_PATIENCE = 1
    FOCAL_LOSS_GAMMA = 3
    NEUTRAL_CLASS_NAME = 'neutral'
    NEUTRAL_CLASS_BOOST_FACTOR = 1.0
    TFIDF_MAX_FEATURES = 1500
    TFIDF_NGRAM_RANGE = (1, 2)
    RANDOM_STATE = 42

config = Config()

# ==============================================================================
# SETUP AND DATA LOADING
# ==============================================================================
print("--- Installing all required libraries ---")
!pip install transformers[torch] scikit-learn pandas sentencepiece --quiet
!pip install lightgbm --quiet
print("✅ Libraries installed.")

# Delete old directory if it exists to ensure a clean run
if os.path.exists(config.OUTPUT_DIR):
    shutil.rmtree(config.OUTPUT_DIR)
os.makedirs(config.OUTPUT_DIR, exist_ok=True)


print("\n--- Step 1: Uploading your AUGMENTED TRAINING set ---")
# ... (data loading code remains the same) ...
LABEL_MAPPING = {'0': 'neutral', 0: 'neutral', '1': 'positive', 1: 'positive', '-1': 'negative', -1: 'negative'}
def standardize_labels(df, label_col='label'):
    str_mapping = {str(k): v for k, v in LABEL_MAPPING.items()}
    df[label_col] = df[label_col].astype(str).replace(str_mapping)
    return df

try:
    uploaded_train = files.upload()
    train_file_name = next(iter(uploaded_train))
    df_train = pd.read_csv(train_file_name)
    df_train = standardize_labels(df_train)
    print(f"\n✅ Augmented training set '{train_file_name}' loaded successfully.")
except Exception as e:
    print(f"\n⚠️ Error loading training file: {e}. Halting.")
    exit()

print("\n--- Step 2: Uploading your ORIGINAL, UNTOUCHED TEST set ---")
try:
    uploaded_test = files.upload()
    test_file_name = next(iter(uploaded_test))
    df_test = pd.read_csv(test_file_name)
    df_test = standardize_labels(df_test)
    print(f"\n✅ Original test set '{test_file_name}' loaded successfully.")
except Exception as e:
    print(f"\n⚠️ Error loading test file: {e}. Halting.")
    exit()

le = LabelEncoder()
all_labels = pd.concat([df_train['label'], df_test['label']], ignore_index=True)
le.fit(all_labels)
X_train_text = df_train['cleaned_text'].astype(str).tolist()
y_train_encoded = le.transform(df_train['label'])
X_test_text = df_test['cleaned_text'].astype(str).tolist()
y_test_encoded = le.transform(df_test['label'])
print("\n✅ Data Loading and Label Standardization Complete.\n")

# ==============================================================================
# FOCAL LOSS & K-FOLD TRAINING
# ==============================================================================
class FocalLoss(torch.nn.Module):
    def __init__(self, alpha=None, gamma=3, reduction='mean'):
        super(FocalLoss, self).__init__(); self.alpha = alpha; self.gamma = gamma; self.reduction = reduction
    def forward(self, inputs, targets):
        ce_loss = F.cross_entropy(inputs, targets, reduction='none'); pt = torch.exp(-ce_loss); focal_loss = (1 - pt)**self.gamma * ce_loss
        if self.alpha is not None: focal_loss = self.alpha[targets] * focal_loss
        return focal_loss.mean() if self.reduction == 'mean' else focal_loss.sum()

print("Starting K-FOLD DeBERTa ensemble training and feature generation...")
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
skf = StratifiedKFold(n_splits=config.N_SPLITS, shuffle=True, random_state=config.RANDOM_STATE)
oof_train_bert_features = np.zeros((len(df_train), 768))
test_bert_feature_predictions = []
tokenizer = AutoTokenizer.from_pretrained(config.MODEL_NAME)

for fold, (train_idx, val_idx) in enumerate(skf.split(X_train_text, y_train_encoded)):
    print(f"\n========== FOLD {fold + 1}/{config.N_SPLITS} ==========")
    X_train_fold, y_train_fold = np.array(X_train_text)[train_idx].tolist(), y_train_encoded[train_idx]
    X_val_fold, y_val_fold = np.array(X_train_text)[val_idx].tolist(), y_train_encoded[val_idx]

    train_encodings = tokenizer(X_train_fold, truncation=True, padding=True, max_length=config.MAX_LENGTH)
    val_encodings = tokenizer(X_val_fold, truncation=True, padding=True, max_length=config.MAX_LENGTH)
    train_dataset = TensorDataset(torch.tensor(train_encodings['input_ids']), torch.tensor(train_encodings['attention_mask']), torch.tensor(y_train_fold))
    val_dataset = TensorDataset(torch.tensor(val_encodings['input_ids']), torch.tensor(val_encodings['attention_mask']), torch.tensor(y_val_fold))
    train_loader = DataLoader(train_dataset, batch_size=config.BATCH_SIZE, shuffle=True)
    val_loader = DataLoader(val_dataset, batch_size=config.BATCH_SIZE, shuffle=False)

    model = AutoModelForSequenceClassification.from_pretrained(config.MODEL_NAME, num_labels=len(le.classes_)).to(device)
    optimizer = AdamW(model.parameters(), lr=config.LEARNING_RATE)
    unique_classes_fold = np.unique(y_train_fold)
    class_weights_balanced = compute_class_weight('balanced', classes=unique_classes_fold, y=y_train_fold)
    custom_class_weights = list(class_weights_balanced)
    neutral_idx = np.where(le.classes_[unique_classes_fold] == config.NEUTRAL_CLASS_NAME)[0]
    if neutral_idx.size > 0:
        custom_class_weights[neutral_idx[0]] *= config.NEUTRAL_CLASS_BOOST_FACTOR

    loss_function = FocalLoss(alpha=torch.tensor(custom_class_weights, dtype=torch.float).to(device), gamma=config.FOCAL_LOSS_GAMMA)
    scheduler = get_linear_schedule_with_warmup(optimizer, num_warmup_steps=0, num_training_steps=len(train_loader) * config.NUM_EPOCHS_KFOLD)

    best_val_loss = float('inf')
    patience_counter = 0
    best_model_state_dict = None # Keep the best model state in memory

    for epoch in range(config.NUM_EPOCHS_KFOLD):
        model.train()
        for batch in train_loader:
            input_ids, attention_mask, labels = [b.to(device) for b in batch]
            optimizer.zero_grad(); outputs = model(input_ids, attention_mask=attention_mask); loss = loss_function(outputs.logits, labels)
            loss.backward(); optimizer.step(); scheduler.step()

        model.eval(); val_loss = 0
        with torch.no_grad():
            for batch in val_loader:
                input_ids, attention_mask, labels = [b.to(device) for b in batch]
                outputs = model(input_ids, attention_mask=attention_mask); val_loss += loss_function(outputs.logits, labels).item()

        avg_val_loss = val_loss / len(val_loader)
        print(f"  Epoch {epoch + 1} | Val Loss: {avg_val_loss:.4f}")

        if avg_val_loss < best_val_loss:
            best_val_loss = avg_val_loss
            patience_counter = 0
            # Instead of saving to disk, copy the state dictionary to a variable
            best_model_state_dict = model.state_dict().copy()
        else:
            patience_counter += 1

        if patience_counter >= config.EARLY_STOPPING_PATIENCE:
            print(f"  -> Stopping early.")
            break

    # Load the best state from memory back into the model
    if best_model_state_dict:
        model.load_state_dict(best_model_state_dict)

    # Define final save directory and save the complete model
    fold_model_dir = os.path.join(config.OUTPUT_DIR, f'fold_{fold+1}')
    os.makedirs(fold_model_dir, exist_ok=True)
    model.save_pretrained(fold_model_dir)
    tokenizer.save_pretrained(fold_model_dir)

    def get_bert_embeddings(texts, model, tokenizer, device):
        model.eval(); all_embeddings = []; base_model = getattr(model, model.base_model_prefix)
        with torch.no_grad():
            for text in texts:
                inputs = tokenizer(text, return_tensors="pt", truncation=True, padding=True, max_length=config.MAX_LENGTH).to(device)
                outputs = base_model(**inputs); all_embeddings.append(outputs.last_hidden_state[:, 0, :].cpu().numpy())
        return np.vstack(all_embeddings)

    oof_train_bert_features[val_idx] = get_bert_embeddings(X_val_fold, model, tokenizer, device)
    test_bert_feature_predictions.append(get_bert_embeddings(X_test_text, model, tokenizer, device))
    print(f"  -> Fold {fold + 1} training, saving, and feature generation complete.")

X_train_bert = oof_train_bert_features; X_test_bert = np.mean(test_bert_feature_predictions, axis=0)
print("\n✅ K-Fold BERT Ensemble Training & Feature Generation Complete.\n")

# ==============================================================================
# HYBRID FEATURES, META-CLASSIFIERS, AND EVALUATION
# ==============================================================================
# ... (This entire section is correct and remains unchanged) ...
print("Starting Final Hybrid Feature Engineering...")
tfidf_vectorizer = TfidfVectorizer(ngram_range=config.TFIDF_NGRAM_RANGE, max_features=config.TFIDF_MAX_FEATURES)
X_train_tfidf = tfidf_vectorizer.fit_transform(X_train_text)
X_test_tfidf = tfidf_vectorizer.transform(X_test_text)
X_train_combined = hstack([X_train_bert, X_train_tfidf]).toarray()
X_test_combined = hstack([X_test_bert, X_test_tfidf]).toarray()
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train_combined)
X_test_scaled = scaler.transform(X_test_combined)
print("✅ Hybrid Feature Engineering Complete.\n")

print("Optimizing and Stacking final classifiers...")
print("\n--- Tuning Base Meta-Classifiers (SVM & LightGBM) ---")
param_grid_svm = {'C': [1, 10, 50], 'kernel': ['rbf'], 'probability': [True]}
grid_search_svm = GridSearchCV(SVC(class_weight='balanced', random_state=config.RANDOM_STATE), param_grid_svm, cv=3, n_jobs=-1, verbose=1)
grid_search_svm.fit(X_train_scaled, y_train_encoded)
best_svm_params = grid_search_svm.best_params_
print(f"Best SVM params: {best_svm_params}")

param_grid_lgbm = {'n_estimators': [100, 200, 300], 'learning_rate': [0.05, 0.1], 'num_leaves': [31, 50]}
grid_search_lgbm = GridSearchCV(lgb.LGBMClassifier(random_state=config.RANDOM_STATE), param_grid_lgbm, cv=3, n_jobs=-1, verbose=1)
grid_search_lgbm.fit(X_train_scaled, y_train_encoded)
best_lgbm_params = grid_search_lgbm.best_params_
print(f"Best LightGBM params: {best_lgbm_params}")

print("\n--- Creating Stacked Ensemble ---")
skf_meta = StratifiedKFold(n_splits=5, shuffle=True, random_state=config.RANDOM_STATE)
oof_svm_preds = np.zeros((len(X_train_scaled), len(le.classes_)))
oof_lgbm_preds = np.zeros((len(X_train_scaled), len(le.classes_)))
test_svm_preds_folds, test_lgbm_preds_folds = [], []
for fold, (train_idx, val_idx) in enumerate(skf_meta.split(X_train_scaled, y_train_encoded)):
    print(f"  Stacking Fold {fold + 1}/5...")
    X_train_fold, X_val_fold = X_train_scaled[train_idx], X_train_scaled[val_idx]; y_train_fold = y_train_encoded[train_idx]
    svm_fold = SVC(**best_svm_params, class_weight='balanced', random_state=config.RANDOM_STATE).fit(X_train_fold, y_train_fold)
    oof_svm_preds[val_idx] = svm_fold.predict_proba(X_val_fold); test_svm_preds_folds.append(svm_fold.predict_proba(X_test_scaled))
    lgbm_fold = lgb.LGBMClassifier(**best_lgbm_params, random_state=config.RANDOM_STATE).fit(X_train_fold, y_train_fold)
    oof_lgbm_preds[val_idx] = lgbm_fold.predict_proba(X_val_fold); test_lgbm_preds_folds.append(lgbm_fold.predict_proba(X_test_scaled))

test_svm_preds = np.mean(test_svm_preds_folds, axis=0); test_lgbm_preds = np.mean(test_lgbm_preds_folds, axis=0)
X_train_stacked = np.concatenate([oof_svm_preds, oof_lgbm_preds], axis=1)
X_test_stacked = np.concatenate([test_svm_preds, test_lgbm_preds], axis=1)

print("\n--- Training Final Stacking Classifier (Logistic Regression) ---")
stacking_classifier = LogisticRegression(random_state=config.RANDOM_STATE)
stacking_classifier.fit(X_train_stacked, y_train_encoded)
stacked_preds = stacking_classifier.predict(X_test_stacked)
stacked_accuracy = accuracy_score(y_test_encoded, stacked_preds)

print("\n--- FINAL ENSEMBLE PERFORMANCE ---")
y_test_original = le.inverse_transform(y_test_encoded)
print(classification_report(y_test_original, le.inverse_transform(stacked_preds), target_names=le.classes_))
print(f"Final Stacked Accuracy: {stacked_accuracy:.4f}")
print("\n✅ Final Model Optimization and Evaluation Complete.\n")

# ==============================================================================
# FINAL SAVING AND **VERIFICATION**
# ==============================================================================
print(f"Saving the final ENSEMBLE model with {stacked_accuracy:.4f} accuracy...")
best_svm = SVC(**best_svm_params, class_weight='balanced', random_state=config.RANDOM_STATE).fit(X_train_scaled, y_train_encoded)
best_lgbm = lgb.LGBMClassifier(**best_lgbm_params, random_state=config.RANDOM_STATE).fit(X_train_scaled, y_train_encoded)

joblib.dump(tfidf_vectorizer, os.path.join(config.OUTPUT_DIR, 'tfidf_vectorizer.joblib'))
joblib.dump(scaler, os.path.join(config.OUTPUT_DIR, 'scaler.joblib'))
joblib.dump(best_svm, os.path.join(config.OUTPUT_DIR, 'svm_base_meta.joblib'))
joblib.dump(best_lgbm, os.path.join(config.OUTPUT_DIR, 'lgbm_base_meta.joblib'))
joblib.dump(stacking_classifier, os.path.join(config.OUTPUT_DIR, 'stacking_final_classifier.joblib'))
joblib.dump(le, os.path.join(config.OUTPUT_DIR, 'label_encoder.joblib'))
print("All non-transformer model components saved.")

# ==================== NEW VERIFICATION STEP ====================
print("\n--- Verifying Saved Model Files Before Zipping ---")
all_folds_ok = True
for i in range(1, config.N_SPLITS + 1):
    fold_dir = os.path.join(config.OUTPUT_DIR, f'fold_{i}')
    print(f"\nChecking contents of {fold_dir}:")
    if os.path.exists(fold_dir):
        files_in_dir = os.listdir(fold_dir)
        print(f" -> Found files: {files_in_dir}")
        # Check for the essential files
        config_ok = 'config.json' in files_in_dir
        weights_ok = 'pytorch_model.bin' in files_in_dir or 'model.safetensors' in files_in_dir
        if config_ok and weights_ok:
            print(f"✅ Fold {i} is complete.")
        else:
            print(f"❌ ERROR: Fold {i} is MISSING essential files! Config OK: {config_ok}, Weights OK: {weights_ok}")
            all_folds_ok = False
    else:
        print(f"❌ FATAL ERROR: Directory for Fold {i} does not exist!")
        all_folds_ok = False
print("\n--- Verification Complete ---")

if not all_folds_ok:
    print("\n\n⚠️ WARNING: One or more folds were not saved correctly. The ZIP file will be incomplete. Please check the logs for errors.")
else:
    print("\n\n✅ All folds verified successfully. Proceeding to zip and download.")
# ==============================================================

# --- Zip and Download ---
shutil.make_archive(config.OUTPUT_DIR.rstrip('/'), 'zip', config.OUTPUT_DIR)
files.download(f"{config.OUTPUT_DIR.rstrip('/')}.zip")
print("\n✅ Model Saving and Download Complete. \n*** ENTIRE SCRIPT FINISHED ***")

--- Installing all required libraries ---
✅ Libraries installed.

--- Step 1: Uploading your AUGMENTED TRAINING set ---


Saving preprocessed_GOLDEN_dataset.csv to preprocessed_GOLDEN_dataset.csv

✅ Augmented training set 'preprocessed_GOLDEN_dataset.csv' loaded successfully.

--- Step 2: Uploading your ORIGINAL, UNTOUCHED TEST set ---


Saving preprocessed_testing_dataset.csv to preprocessed_testing_dataset.csv

✅ Original test set 'preprocessed_testing_dataset.csv' loaded successfully.

✅ Data Loading and Label Standardization Complete.

Starting K-FOLD DeBERTa ensemble training and feature generation...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/52.0 [00:00<?, ?B/s]

...

/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(



--- Training Final Stacking Classifier (Logistic Regression) ---

--- FINAL ENSEMBLE PERFORMANCE ---
              precision    recall  f1-score   support

    negative       1.00      0.89      0.94        54
     neutral       0.85      1.00      0.92        57
    positive       1.00      0.92      0.96        50

    accuracy                           0.94       161
   macro avg       0.95      0.94      0.94       161
weighted avg       0.95      0.94      0.94       161

Final Stacked Accuracy: 0.9379

✅ Final Model Optimization and Evaluation Complete.

Saving the final ENSEMBLE model with 0.9379 accuracy...
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.319235 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 202675
[LightGBM] [Info] Number of data points in the train set: 10372, number of used features: 2267
[LightGBM] [Info] Start training from score -1.098419
[LightGBM] [Info] Start train

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>


✅ Model Saving and Download Complete. 
*** ENTIRE SCRIPT FINISHED ***
